In [2]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS

SEED, L = 11, 16
torch.manual_seed(SEED); np.random.seed(SEED)

def load(split):
    z = np.load(DATA / "windows" / f"{split}.npz", allow_pickle=True)
    d = {k: z[k] for k in z.files}
    d["attack_type"] = d["attack_type"].astype(str)
    return d

tr, va, tu, te = load("train"), load("val"), load("tune"), load("test")
D = tr["X"].shape[2]
print(tr["X"].shape, va["X"].shape, tu["X"].shape, te["X"].shape)

(100000, 16, 28) (30000, 16, 28) (40551, 16, 28) (109518, 16, 28)


In [3]:
class Block(nn.Module):
    def __init__(self, ch, dil):
        super().__init__()
        self.c1 = nn.Conv1d(ch, ch, 3, padding=dil, dilation=dil)
        self.c2 = nn.Conv1d(ch, ch, 3, padding=dil, dilation=dil)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(x + self.c2(self.act(self.c1(x))))

class TCNMAE(nn.Module):
    def __init__(self, d, ch=64, dils=(1, 2, 4)):
        super().__init__()
        self.inp = nn.Conv1d(d + 2, ch, 1)
        self.blocks = nn.Sequential(*[Block(ch, k) for k in dils])
        self.out = nn.Conv1d(ch, d, 1)
    def forward(self, x, valid, masked):
        xin = x * (valid & ~masked).unsqueeze(-1)
        pad = (~valid).float().unsqueeze(-1)
        m = masked.float().unsqueeze(-1)
        h = torch.cat([xin, pad, m], -1).transpose(1, 2)
        return self.out(self.blocks(self.inp(h))).transpose(1, 2)

def random_mask(valid, p=0.25):
    r = torch.rand(valid.shape)
    r[~valid] = 2.0
    masked = (r < p) & valid
    force = r.argmin(1)                                  # make sure every window masks something
    masked[torch.arange(len(valid)), force] = True
    return masked & valid

def masked_loss(model, x, valid, masked):
    err = ((model(x, valid, masked) - x) ** 2).mean(-1)
    return (err * masked).sum() / masked.sum()

Xtr = torch.from_numpy(tr["X"]); Vtr = torch.from_numpy(tr["mask"])
Xva = torch.from_numpy(va["X"]); Vva = torch.from_numpy(va["mask"])
g = torch.Generator().manual_seed(SEED)
Mva = random_mask(Vva)                                   # fixed masks, so early stopping is stable

model = TCNMAE(D)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print("parameters:", sum(p.numel() for p in model.parameters()))

best, best_state, bad = float("inf"), None, 0
for epoch in range(25):
    t0 = time.time()
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 512):
        idx = perm[i:i + 512]
        x, v = Xtr[idx], Vtr[idx]
        opt.zero_grad()
        masked_loss(model, x, v, random_mask(v)).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        vl = np.mean([masked_loss(model, Xva[i:i + 2048], Vva[i:i + 2048], Mva[i:i + 2048]).item()
                      for i in range(0, len(Xva), 2048)])
    print(f"epoch {epoch + 1:2d}  val loss {vl:.4f}  ({time.time() - t0:.0f}s)")
    if vl < best - 1e-4:
        best, bad = vl, 0
        best_state = {k: t.clone() for k, t in model.state_dict().items()}
    else:
        bad += 1
        if bad >= 3:
            break
model.load_state_dict(best_state)
torch.save(model.state_dict(), RESULTS / "m4_tcn_seed11.pt")

parameters: 77916
epoch  1  val loss 0.4409  (17s)
epoch  2  val loss 0.4242  (18s)
epoch  3  val loss 0.4198  (19s)
epoch  4  val loss 0.4155  (17s)
epoch  5  val loss 0.4135  (16s)
epoch  6  val loss 0.4183  (15s)
epoch  7  val loss 0.4077  (15s)
epoch  8  val loss 0.4165  (17s)
epoch  9  val loss 0.4069  (16s)
epoch 10  val loss 0.4043  (16s)
epoch 11  val loss 0.4058  (16s)
epoch 12  val loss 0.4042  (16s)
epoch 13  val loss 0.4094  (16s)


In [4]:
@torch.no_grad()
def window_scores(X, V, K=4, bs=2048):
    model.eval()
    X, V = torch.from_numpy(X), torch.from_numpy(V)
    pos = torch.arange(L)
    out = []
    for i in range(0, len(X), bs):
        x, v = X[i:i + bs], V[i:i + bs]
        tot = torch.zeros(len(x))
        for k in range(K):
            masked = (pos % K == k).unsqueeze(0) & v
            tot += (((model(x, v, masked) - x) ** 2).mean(-1) * masked).sum(1)
        out.append((tot / v.sum(1)).numpy())
    return np.concatenate(out)

def to_df(split, d):
    return pd.DataFrame({"id": d["win_ids"], "score": window_scores(d["X"], d["mask"]),
                         "label": d["label"].astype(int), "attack_type": d["attack_type"], "split": split})

val_s, tune_s, test_s = to_df("val", va), to_df("tune", tu), to_df("test", te)

keys = ["auroc", "tpr@fpr0.01", "real_fpr@0.01", "tpr@fpr0.05", "macro_tpr@fpr0.05", "real_fpr@0.05"]
for name, d_ in [("tune", tune_s), ("test", test_s)]:
    r, o = evaluate(d_, val_df=val_s), evaluate(d_)
    print(name, "val-threshold:", {k: round(r[k], 3) for k in keys})
    print(name, "ORACLE:", {k: round(o[k], 3) for k in ["auroc", "tpr@fpr0.01", "tpr@fpr0.05"]})
    print("   per-attack TPR@1%:", {k: round(v, 3) for k, v in r["per_attack_tpr@fpr0.01"].items()})

pd.concat([val_s, tune_s, test_s]).to_parquet(RESULTS / "scores" / f"m4_tcn_nftoniot_seed{SEED}.parquet")

tune val-threshold: {'auroc': 0.571, 'tpr@fpr0.01': 0.137, 'real_fpr@0.01': 0.05, 'tpr@fpr0.05': 0.271, 'macro_tpr@fpr0.05': 0.58, 'real_fpr@0.05': 0.106}
tune ORACLE: {'auroc': 0.571, 'tpr@fpr0.01': 0.041, 'tpr@fpr0.05': 0.137}
   per-attack TPR@1%: {'ddos': 0.304, 'dos': 0.301, 'injection': 0.302, 'mitm': 0.0, 'password': 0.47, 'scanning': 0.049, 'xss': 0.528}
test val-threshold: {'auroc': 0.48, 'tpr@fpr0.01': 0.399, 'real_fpr@0.01': 0.378, 'tpr@fpr0.05': 0.671, 'macro_tpr@fpr0.05': 0.687, 'real_fpr@0.05': 0.708}
test ORACLE: {'auroc': 0.48, 'tpr@fpr0.01': 0.0, 'tpr@fpr0.05': 0.002}
   per-attack TPR@1%: {'backdoor': 0.232, 'ddos': 0.36, 'dos': 0.809, 'injection': 0.145, 'mitm': 0.555, 'password': 0.199, 'ransomware': 0.151, 'scanning': 0.129, 'xss': 0.71}
